# Profilierung regionaler Netzlast und Ausfälle mit PROC CHART

## Zusammenfassung

Ein Netzbetriebsanalyst nutzt PROC CHART, um eine synthetische Stichprobe von Zählerablesungen an Abgangsstromkreisen über vier Versorgungsregionen und vier Erzeugungsquellen zu profilieren. Das Notebook geht vertikale Balken-, horizontale Balken-, Block- und Kreisdiagramme durch, um den Erzeugungsmix zusammenzufassen, mittlere und Gesamtlast je Region zu vergleichen, die abendliche Nachfragespitze nach Stunde offenzulegen und Ausfallminuten nach Quelle zu ranken — die Art schneller, textbasierter Exploration, die ein Energie- und Versorgungsteam durchführt, bevor es sich auf einen grafischen Bericht festlegt. Der DATA-Step fordert 1,200 Zeilen an; dieses Notebook wurde im unlizenzierten Modus gerendert, der die Arbeitsdatenmenge auf die ersten 100 Ablesungen begrenzt, sodass jedes Diagramm unten diese 100-Zeilen-Stichprobe zusammenfasst.

## Datenquellen

| Datenmenge | Zeilen | Beschreibung |
|---------|------|-------------|
| `grid_ops` | 100 (synthetische Stichprobe) | Eine Zeile je Zählerablesung an einem Abgangsstromkreis in einem regionalen Netz, inline mit `call streaminit(20260531)` und `rand()` erzeugt. Die DATA-Step-Schleife fordert 1,200 Ablesungen an, doch der unlizenzierte Modus begrenzt die gespeicherte Datenmenge auf 100 Beobachtungen, sodass die Diagramme unten diese 100 Zeilen beschreiben. |

# Profilierung regionaler Netzlast und Ausfälle mit PROC CHART

PROC CHART erzeugt zeichenbasierte Balken-, Block- und Kreisdiagramme direkt im Listing — ohne ein ODS-Graphics-Gerät. Das macht es zu einem schnellen Erst-Explorationswerkzeug für ein Netzbetriebsteam, das die Form seiner Last- und Zuverlässigkeitsdaten *sehen* möchte, bevor es aufwendige GCHART- oder SGPLOT-Visualisierungen erstellt.

In diesem Notebook:

1. erzeugen wir einen synthetischen Monat an Zählerablesungen von Abgangsstromkreisen für einen regionalen Netzbetreiber.
2. stellen wir den **Erzeugungsmix** dar (Anteil der Ablesungen nach Quelle).
3. vergleichen wir **mittlere und gesamte gelieferte Last** über die Versorgungsregionen.
4. legen wir die **abendliche Nachfragespitze** nach Tagesstunde offen.
5. verwenden wir ein **Blockdiagramm**, um Region gegen Erzeugungsquelle zu kreuzen.
6. ranken wir **Ausfallminuten** nach Quelle, um die unzuverlässigste Einspeisung zu finden.

Jede Anweisung und Option unten ist Standard-SAS-9.4-PROC-CHART-Syntax.

## Schritt 1 — Erzeugung der synthetischen Betriebsdaten

Der DATA-Step unten fabriziert Zählerablesungen in einer Schleife von 1,200 Iterationen. Jeder Ablesung werden eine Versorgungsregion, eine Erzeugungsquelle (Gas dominiert, Wind, Solar und Wasser machen den Rest aus) und eine Tagesstunde zugewiesen. Die Last ist im abendlichen Fenster von 17:00–21:00 höher (und wir kennzeichnen diese Ablesungen als Spitze), und wir ziehen Ausfallminuten aus einer Poisson-Verteilung. `call streaminit` legt den Zufalls-Seed fest, sodass die Daten reproduzierbar sind.

Der NOTE-Hinweis im Log zeigt das praktische Ergebnis: Dieser Lauf erfolgt im unlizenzierten Modus, sodass die gespeicherte Datenmenge `grid_ops` auf die ersten 100 dieser Ablesungen begrenzt ist (100 Zeilen, 7 Spalten). Jedes folgende Diagramm fasst diese 100-Zeilen-Stichprobe zusammen, und jedes PROC-CHART-Log bestätigt, dass 100 Zeilen gelesen wurden.

In [1]:
/* Synthetische Abgangsstromkreis-Betriebsdaten für einen regionalen Netzbetreiber */
DATEN grid_ops;
    AUFRUFEN streaminit(20260531);
    LÄNGE region $12 source $9 peak_flag $4;
    FELD regions[4] $12 _temporary_
        ('Nord','Sued','Ost','West');
    AUSFÜHRUNG meter_id = 1 BIS 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        WENN u < 0.42 DANN source = 'Gas';
        SONST WENN u < 0.70 DANN source = 'Wind';
        SONST WENN u < 0.88 DANN source = 'Solar';
        SONST source = 'Wasser';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'Nord') + 3 * (region = 'West');
        WENN hour >= 17 UND hour <= 21 DANN AUSFÜHRUNG;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Ja';
        ENDE;
        SONST AUSFÜHRUNG;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'Nein';
        ENDE;
        WENN load_mwh < 0 DANN load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        AUSGABE;
    ENDE;
    ENTFERNEN r u BASE;
AUSFÜHREN;


NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.23 seconds
  cpu   0.23 seconds


## Schritt 2 — Erzeugungsmix

Welchen Anteil der Ablesungen macht jede Erzeugungsquelle aus? Ein vertikales Balkendiagramm mit `TYPE=PERCENT` beantwortet dies direkt: Die Balkenhöhen sind der Prozentsatz aller Beobachtungen, die in jede Quellenkategorie fallen. Da `source` eine Zeichenvariable ist, behandelt PROC CHART ihre Werte automatisch als diskrete Kategorien.

In [2]:
PROZEDUR chart DATEN=grid_ops;
    VBAR source / type=PERCENT;
    BEZEICHNUNG source="Energiequelle";
    TITEL "Erzeugungsmix – Anteil der Ablesungen nach Quelle";
AUSFÜHREN;


Percent of Energiequelle

         |                ******        
         |                ******        
   40.00 +                ******        
         |                ******        
         |                ******        
   30.00 +                ******        
         |        ******  ******        
         |        ******  ******        
   20.00 +        ******  ******        
         |        ******  ******        
         |******  ******  ******        
         |******  ******  ******  ******
   10.00 +******  ******  ******  ******
         |******  ******  ******  ******
         |******  ******  ******  ******
         |                              
         +------------------------------+
          Wasser   Wind    Gas    Solar 
                   Energiequelle





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 3 — Mittlere gelieferte Last je Region

Nun wechseln wir vom Zählen zum Zusammenfassen einer Messgröße. Die Angabe von `load_mwh` in `SUMVAR=` mit `TYPE=MEAN` macht die Balkenlänge zur durchschnittlichen Last je Region, und wir fordern die Statistikspalten explizit an: `MEAN` gibt den Durchschnitt neben jedem Balken aus und `CFREQ` fügt eine kumulative Häufigkeitsspalte hinzu. Nord trägt die höchste durchschnittliche gelieferte Last (Mittelwert etwa 28.0 MWh), im Einklang mit dem in die Daten eingebauten regionalen Versatz, während Sued am niedrigsten liegt (etwa 19.8 MWh).

Wir übergeben außerdem `ASCENDING`, um die Balken vom niedrigsten zum höchsten Mittelwert zu ordnen. Diese PROC-CHART-Implementierung setzt das um: In der Ausgabe erscheinen die Balken in aufsteigender Reihenfolge der mittleren Last — Sued (19.8), Ost (21.7), West (24.2), Nord (28.0) —, sodass die Region mit der geringsten durchschnittlichen Last oben und Nord mit der höchsten unten steht. Die Spalte `Mean` gibt den Zahlenwert neben jedem Balken aus, sodass sich die Rangfolge auch direkt ablesen lässt.

In [3]:
PROZEDUR chart DATEN=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
    BEZEICHNUNG region="Region" load_mwh="Last (MWh)";
    TITEL "Mittlere gelieferte Last je Region";
AUSFÜHREN;


Mean of Region

Region                                                  Mean           N     Percent
                                                                                    
  Sued  ****************************                   19.80       19.80       21.00
   Ost  *******************************                21.72       41.52       29.00
  West  **********************************             24.17       65.69       23.00
  Nord  ****************************************       28.03       93.72       27.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 4 — Gesamtlast je Region

Hier macht `TYPE=SUM` jeden Balken zur *Gesamt*last der Region statt zum Durchschnitt, sodass die höheren Balken die Regionen kennzeichnen, die über die Stichprobenablesungen die meiste aggregierte Energie liefern. Nord führt erneut bei der gesamten gelieferten Last.

Die Anweisung fordert außerdem `SUBGROUP=peak_flag` an und weist PROC CHART an, jeden Balken danach aufzuteilen, ob die Ablesung in das abendliche Spitzenfenster fiel. PROC CHART segmentiert jeden Balken mit einem eigenen Glyph je Untergruppenebene — `J` für die Spitzenablesungen (`Ja`) und `N` für die Nebenzeit (`Nein`) — und gibt darunter eine Legende aus, die jedes Symbol seiner `peak_flag`-Ebene zuordnet. So lässt sich für jede Region ablesen, welcher Anteil der gesamten gelieferten Last in das abendliche Spitzenfenster fällt. Eine feinere, stundengenaue Ansicht der Spitze folgt in Schritt 5.

In [4]:
PROZEDUR chart DATEN=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
    BEZEICHNUNG region="Region" load_mwh="Last (MWh)" peak_flag="Spitze";
    TITEL "Gesamtlast je Region nach Spitzenkennzeichen";
AUSFÜHREN;


Sum of Region

         |                     JJJJJ
         |                     JJJJJ
         |                     JJJJJ
     600 +              JJJJJ  JJJJJ
         |JJJJJ         JJJJJ  JJJJJ
         |JJJJJ         JJJJJ  JJJJJ
         |JJJJJ         JJJJJ  JJJJJ
     400 +NNNNN  JJJJJ  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
     200 +NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |                          
         +--------------------------+
          West   Sued    Ost   Nord 
                    Region

Symbol  Spitze
------  ------
  N     Nein  
  J     Ja    





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 5 — Lastverlauf über den Tag

`hour` ist kontinuierlich, daher legen wir die Klassenbildung selbst mit einer expliziten `MIDPOINTS=`-Liste an 4-Stunden-Zentren (2, 6, 10, 14, 18, 22) fest. Jeder Balken zeigt die mittlere gelieferte Last für Ablesungen nahe dieser Stunde. Der auf 18 zentrierte Balken sollte hervorstechen — das ist die abendliche Nachfragespitze, die wir in die Daten eingebaut haben.

In [5]:
PROZEDUR chart DATEN=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
    BEZEICHNUNG hour="Tagesstunde" load_mwh="Last (MWh)";
    TITEL "Mittlere Last nach Tagesstunde";
AUSFÜHREN;


Mean of Tagesstunde

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                         Tagesstunde





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 6 — Region nach Erzeugungsquelle (Blockdiagramm)

Ein BLOCK-Diagramm stellt eine kleine Zweiwegetabelle als Raster von Blöcken dar. Mit `GROUP=source` und `SUMVAR=load_mwh / TYPE=MEAN` kreuzt das Diagramm jede Region gegen die sie versorgende Erzeugungsquelle, wobei die Blockhöhe proportional zur mittleren Last ist — eine kompakte Möglichkeit, zu erkennen, welche Region/Quelle-Kombinationen die höchste durchschnittliche Last tragen.

In [6]:
PROZEDUR chart DATEN=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
    BEZEICHNUNG region="Region" source="Energiequelle" load_mwh="Last (MWh)";
    TITEL "Mittlere Last: Region nach Energiequelle";
AUSFÜHREN;


Block chart of Mean of Region by Energiequelle

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West    Sued    Ost     Nord 
             Region





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 7 — Erzeugungsmix als Kreisdiagramm

Dieselbe Information zum Quellenanteil aus Schritt 2, als Kreisdiagramm gezeichnet. PIE mit `TYPE=PERCENT` bemisst jedes Segment nach seinem Prozentsatz der Gesamtablesungen und gibt neben der Abbildung eine Legende der Segmentprozentsätze aus.

In [7]:
PROZEDUR chart DATEN=grid_ops;
    PIE source / type=PERCENT;
    BEZEICHNUNG source="Energiequelle";
    TITEL "Erzeugungsmix als Kreisdiagramm";
AUSFÜHREN;


Pie chart of Energiequelle

   Energiequelle     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
          Wasser       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Schritt 8 — Ausfallminuten nach Quelle

Schließlich ranken wir die Zuverlässigkeit. `SUMVAR=outage_min` mit `TYPE=SUM` summiert die Ausfallminuten je Quelle. Wir übergeben `DESCENDING`, um die am schlechtesten abschneidende Quelle nach oben zu bringen. Anders als `ASCENDING` in Schritt 3 ist `DESCENDING` in dieser PROC-CHART-Implementierung noch nicht verdrahtet, sodass die Balken hier in Kategorienreihenfolge ausgegeben werden (Wasser, Wind, Gas, Solar) statt vom höchsten zum niedrigsten Wert. Lesen Sie die Rangfolge daher aus der ausgegebenen Spalte `Sum`: Gas verursacht in dieser Stichprobe die meisten gesamten Ausfallminuten (122), deutlich vor Wind (64), Solar (43) und Wasser (38). Das folgt dem Erzeugungsmix — Gas bedient die meisten Ablesungen, sodass es insgesamt die meisten Ausfallminuten anhäuft.

In [8]:
PROZEDUR chart DATEN=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum descending;
    BEZEICHNUNG source="Energiequelle" outage_min="Ausfallminuten";
    TITEL "Ausfallminuten nach Energiequelle";
AUSFÜHREN;


Sum of Energiequelle

Energiequelle                                                   Sum        Cum.     Percent
                                                                            Sum            
       Wasser  ************                                      38          38       14.00
         Wind  *********************                             64         102       28.00
          Gas  ****************************************         122         224       45.00
        Solar  **************                                    43         267       13.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpretation der Ergebnisse

Die Diagramme gemeinsam gelesen liefern dem Betriebsteam ein schnelles Lagebild (über die 100-Zeilen-Stichprobe, die dieser Lauf erfasst hat):

- **Erzeugungsmix (Schritte 2 und 7).** Gas trägt den größten Anteil der Ablesungen (etwa 45%), Wind an zweiter Stelle (etwa 28%) sowie Wasser und Solar dahinter (etwa 14% und 13%) — das vertikale Balken- und das Kreisdiagramm erzählen dieselbe Geschichte auf zwei Weisen, eine nützliche Plausibilitätsprüfung.
- **Last je Region (Schritte 3 und 4).** Der HBAR der mittleren Last zeigt Nord mit der höchsten durchschnittlichen gelieferten Last (Mittelwert ~28 MWh) und Sued mit der niedrigsten (~20 MWh), im Einklang mit dem in die Daten eingebauten regionalen Versatz. Das SUM-Diagramm bestätigt, dass Nord auch die meiste Gesamtenergie liefert.
- **Täglicher Lastverlauf (Schritt 5).** Der auf Stunde 18 zentrierte Mittelpunktbalken ist eindeutig der höchste und bestätigt die in die Daten eingebaute Nachfragespitze von 17:00–21:00 — genau dort, wo ein Versorger Demand-Response und Kapazitätsplanung fokussieren würde.
- **Zuverlässigkeit (Schritt 8).** Die Summierung der Ausfallminuten nach Quelle bringt Gas als größten Beitragenden zur Ausfallzeit in dieser Stichprobe hervor (122 Minuten), das natürliche nächste Ziel für eine Wartungsprüfung — obwohl das größtenteils widerspiegelt, dass Gas die meisten Ablesungen bedient.

Ein Hinweis zu den Darstellungsoptionen: Das aufsteigende Umordnen der Balken mit `ASCENDING` (Schritt 3) und die Balkensegmentierung mit `SUBGROUP=` (Schritt 4) werden von dieser PROC-CHART-Implementierung gerendert. Das absteigende Umordnen mit `DESCENDING` (Schritt 8) wird dagegen vom Parser akzeptiert, aber noch nicht umgesetzt, sodass die dortige Rangfolge aus der ausgegebenen Spalte `Sum` statt aus der Balkenreihenfolge abgelesen wird.

PROC CHART liefert nur Zeichenausgabe, daher würde das Team für präsentationsreife Visualisierungen dieselben Ansichten auf PROC GCHART oder PROC SGPLOT übertragen. Doch als aufwandsfreier erster Durchgang über einen frischen Extrakt beantworten diese Diagramme die betrieblichen Fragen — Mix, Größenordnung und Timing — in Sekunden.